### Script for saving pressure-level CMIP6 data subsetted/averaged over the WUS

In [2]:
import netCDF4
#import h5netcdf
import xarray as xr
import scipy
import os
os.environ['ESMFMKFILE'] = '/home/tessj/miniconda3/envs/xarrayxesmf/lib/esmf.mk'
import xesmf as xe
import numpy as np
#import xeofs as xeo
import scipy.stats as stats
import warnings
import matplotlib.pyplot as plt
import scipy.signal as signal
import pandas as pd
import cartopy.crs as ccrs
import cartopy
import seaborn as sns

def fix_coords_lon(ds):
    ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180)).sortby(['longitude','latitude'])
    return ds

def fix_coords_lon_lat(ds):
    ds = ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180)).sortby(['lon','lat'])
    return ds

wusbox = [-125, -102, 32,  49]

In [3]:
def weighted_cov_corr(da_a, da_b, weights, dim = None, method = None):
    """
    Internal method for weighted cov or cor.
    """
    # 1. Broadcast the two arrays
    da_a, da_b = xr.align(da_a, da_b, join="inner", copy=False)

    # 2. Ignore the nans
    valid_values = da_a.notnull() & da_b.notnull()
    da_a = da_a.where(valid_values)
    da_b = da_b.where(valid_values)
    #valid_count = valid_values.sum(dim) - ddof

    # 3. Detrend along the given dim
    demeaned_da_a = da_a - da_a.weighted(weights).mean(dim=dim)
    demeaned_da_b = da_b - da_b.weighted(weights).mean(dim=dim)

    # 4. Compute covariance along the given dim
    # N.B. `skipna=True` is required or auto-covariance is computed incorrectly. E.g.
    # Try xr.cov(da,da) for da = xr.DataArray([[1, 2], [1, np.nan]], dims=["x", "time"])
    cov = (demeaned_da_a.conj() * demeaned_da_b).weighted(weights).mean(dim=dim, skipna=True)

    if method == "cov":
        return cov
    else:
        # compute std + corr
        da_a_std = da_a.weighted(weights).std(dim=dim)
        da_b_std = da_b.weighted(weights).std(dim=dim)
        corr = cov / (da_a_std * da_b_std)
        return corr

In [4]:
import statsmodels.api as sm
import json

In [5]:
with open('/home/tessj/wus_temp_trends/modmembers_geq3_tas.json', 'r') as file:
        modmembers_geq3_tas = json.load(file)

In [12]:
var='ta'

modmembers_historical = {}
for model in list(modmembers_geq3_tas.keys())[:]:
    model_directories = os.listdir(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{model}/')

    members = [filename[len(model + f'{var}_Amon_historical__'):-len('_19800101-20141231.nc')] for filename in model_directories if (filename[:len(var)]==var) & ('historical' in filename)]
    modmembers_historical[model] = list(set([member for member in members if member[0] == 'r']))
    
modmembers_ssp = {}
for model in list(modmembers_geq3_tas.keys())[:]:
    model_directories = os.listdir(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{model}/')
    members = [filename[len(model + f'{var}_Amon_ssp245__'):-len('_20150101-20241231.nc')] for filename in model_directories if (filename[:len(var)]==var) & ('ssp245' in filename)]
    modmembers_ssp[model] = list(set([member for member in members if member[0] == 'r']))

modmembers_ta= {}
for model in list(modmembers_historical.keys()):
    if len((set(modmembers_historical[model]) & set(modmembers_ssp[model])))>0:
        modmembers_ta[model] = list(set(modmembers_historical[model]) & set(modmembers_ssp[model]))

In [13]:
var='hus'

modmembers_historical = {}
for model in list(modmembers_geq3_tas.keys())[:]:
    model_directories = os.listdir(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{model}/')

    members = [filename[len(model + f'{var}_Amon_historical__'):-len('_19800101-20141231.nc')] for filename in model_directories if (filename[:len(var)]==var) & ('historical' in filename)]
    modmembers_historical[model] = list(set([member for member in members if member[0] == 'r']))
    
modmembers_ssp = {}
for model in list(modmembers_geq3_tas.keys())[:]:
    model_directories = os.listdir(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{model}/')
    members = [filename[len(model + f'{var}_Amon_ssp245__'):-len('_20150101-20241231.nc')] for filename in model_directories if (filename[:len(var)]==var) & ('ssp245' in filename)]
    modmembers_ssp[model] = list(set([member for member in members if member[0] == 'r']))

modmembers_hus= {}
for model in list(modmembers_historical.keys()):
    if len((set(modmembers_historical[model]) & set(modmembers_ssp[model])))>0:
        modmembers_hus[model] = list(set(modmembers_historical[model]) & set(modmembers_ssp[model]))

In [14]:
var='zg'

modmembers_historical = {}
for model in list(modmembers_geq3_tas.keys())[:]:
    model_directories = os.listdir(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{model}/')

    members = [filename[len(model + f'{var}_Amon_historical__'):-len('_19800101-20141231.nc')] for filename in model_directories if (filename[:len(var)]==var) & ('historical' in filename)]
    modmembers_historical[model] = list(set([member for member in members if member[0] == 'r']))
    
modmembers_ssp = {}
for model in list(modmembers_geq3_tas.keys())[:]:
    model_directories = os.listdir(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{model}/')
    members = [filename[len(model + f'{var}_Amon_ssp245__'):-len('_20150101-20241231.nc')] for filename in model_directories if (filename[:len(var)]==var) & ('ssp245' in filename)]
    modmembers_ssp[model] = list(set([member for member in members if member[0] == 'r']))

modmembers_zg= {}
for model in list(modmembers_historical.keys()):
    if len((set(modmembers_historical[model]) & set(modmembers_ssp[model])))>0:
        modmembers_zg[model] = list(set(modmembers_historical[model]) & set(modmembers_ssp[model]))

In [15]:
var='ts'

modmembers_historical = {}
for model in list(modmembers_geq3_tas.keys())[:]:
    model_directories = os.listdir(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{model}/')

    members = [filename[len(model + f'{var}_Amon_historical__'):-len('_19800101-20141231.nc')] for filename in model_directories if (filename[:len(var)]==var) & ('historical' in filename)]
    modmembers_historical[model] = list(set([member for member in members if member[0] == 'r']))
    
modmembers_ssp = {}
for model in list(modmembers_geq3_tas.keys())[:]:
    model_directories = os.listdir(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{model}/')
    members = [filename[len(model + f'{var}_Amon_ssp245__'):-len('_20150101-20241231.nc')] for filename in model_directories if (filename[:len(var)]==var) & ('ssp245' in filename)]
    modmembers_ssp[model] = list(set([member for member in members if member[0] == 'r']))

modmembers_ts= {}
for model in list(modmembers_historical.keys()):
    if len((set(modmembers_historical[model]) & set(modmembers_ssp[model])))>0:
        modmembers_ts[model] = list(set(modmembers_historical[model]) & set(modmembers_ssp[model]))

In [16]:
modmembers_allvars = {}
for model in list(modmembers_hus.keys()):
    if len((set(modmembers_ta[model]) & set(modmembers_hus[model])) & (set(modmembers_zg[model]) & set(modmembers_ts[model])))>0:
        modmembers_allvars[model] = list((set(modmembers_ta[model]) & set(modmembers_hus[model])) & (set(modmembers_zg[model]) & set(modmembers_ts[model])))
        

In [17]:
incompletes = {'CNRM-CM6-1': ['r7i1p1f2', 'r8i1p1f2', 'r10i1p1f2', 'r9i1p1f2']}

for k, v in modmembers_allvars.items():
    if k in incompletes.keys():
        modmembers_allvars[k] = [x for x in modmembers_allvars[k] if x not in incompletes[k]]

In [6]:
models_rldscs = os.listdir('/d5/tessj/data/CMIP6/cmip6-esgf-globus/historical-processed/')
modmembers_rldscs = {}
modgr_rldscs = {}


for model in models_rldscs[:]:
    members_hist = [file.rsplit('_')[-2] for file in os.listdir(f'/d5/tessj/data/CMIP6/cmip6-esgf-globus/historical-processed/{model}')]
    members_ssp245 = [file.rsplit('_')[-2] for file in os.listdir(f'/d5/tessj/data/CMIP6/cmip6-esgf-globus/ssp245-processed/{model}')]
    members = list(set(members_hist) & set(members_ssp245))
    modmembers_rldscs[model] = members

In [7]:
with open('/home/tessj/wus_temp_trends/modmembers_rlds_hist.json', 'r') as file:
        modmembers_rlds_hist = json.load(file)
with open('/home/tessj/wus_temp_trends/modmembers_rlds_ssp245.json', 'r') as file:
        modmembers_rlds_ssp245 = json.load(file)
        
modmembers_lwd= {}
for model in list(modmembers_rlds_hist.keys()):
    if ((model in modmembers_rlds_hist.keys()) & (model in modmembers_rlds_ssp245.keys())):
        if len(set(modmembers_rlds_hist[model]) & set(modmembers_rlds_ssp245[model]))>0:
            modmembers_lwd[model] = list(set(modmembers_rlds_hist[model]) & set(modmembers_rlds_ssp245[model]))
            
            
incomplete_runs_lwd_ssp245  =    {'CNRM-CM6-1': ['r9i1p1f2', 'r7i1p1f2', 'r10i1p1f2', 'r8i1p1f2'],
 'UKESM1-0-LL': ['r16i1p1f2',
  'r9i1p1f2',
  'r11i1p1f2',
  'r18i1p1f2',
  'r10i1p1f2',
  'r12i1p1f2',
  'r19i1p1f2',
  'r17i1p1f2'],
 'HadGEM3-GC31-LL': ['r3i1p1f3', 'r2i1p1f3', 'r4i1p1f3']}

for k, v in modmembers_lwd.items():
    if k in incomplete_runs_lwd_ssp245.keys():
        modmembers_lwd[k] = [x for x in modmembers_lwd[k] if x not in incomplete_runs_lwd_ssp245[k]]

modmembers_lwd = {k:modmembers_lwd[k] for k in models_rldscs if k in modmembers_lwd.keys()}

In [18]:
json.dump(modmembers_allvars, open('/home/tessj/wus_temp_trends/modmembers_allvars_WUS_subset_forpyrte.json', 'w')) 

In [19]:
landmask_era5 = fix_coords_lon(xr.open_dataarray('/home/tessj/wildfire-clim/land_sea_mask.nc').isel(time=-1).load()).rename({'latitude':'lat', 'longitude':'lon'}).drop(['expver','time'])


In [21]:
times = pd.date_range(start="1980-01-01", end="2024-12-31", freq="MS")
times

DatetimeIndex(['1980-01-01', '1980-02-01', '1980-03-01', '1980-04-01',
               '1980-05-01', '1980-06-01', '1980-07-01', '1980-08-01',
               '1980-09-01', '1980-10-01',
               ...
               '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01',
               '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01',
               '2024-11-01', '2024-12-01'],
              dtype='datetime64[ns]', length=540, freq='MS')

In [79]:
plevs = np.array([  1000.,   5000.,  10000.,  15000.,  20000.,  25000.,  30000.,  40000.,
        50000.,  60000.,  70000.,  85000.,  92500., 100000.])

In [85]:
for var in ['ta', 'hus', 'zg', 'ts'][3:]:    
    print(var)
    models_var_avg_list = []

    for model in list(modmembers_allvars.keys()):
        print(model, len(modmembers_allvars[model]))
        model_var_avg_list = []


        for member_id in modmembers_allvars[model][:]:
            ds_hist = xr.open_dataset(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{model}/{var}_Amon_{model}_historical_{member_id}_19800101-20141231.nc').sel(time=slice('1980-01-01', '2014-12-30'))
            ds_ss245 = xr.open_dataset(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{model}/{var}_Amon_{model}_ssp245_{member_id}_20150101-20241231.nc')
            ds_total = xr.concat([ds_hist, ds_ss245], dim='time')
            #break
            #if len(ds_total.time) != len(times): 
            #    print(model, member_id)

            landmask = landmask_era5.interp_like(ds_hist)
            landmask = landmask.where(landmask>0.4, np.nan)
            landmask = landmask/landmask
            weights = np.cos(np.deg2rad(landmask.lat))

            ds_wus = ds_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
            ds_wus_mean = ds_wus.weighted(weights).mean(['lon','lat'])
            model_var_avg_list.append(ds_wus_mean.assign_coords({'member_id':member_id, 'time':times}))

        model_var_avg_ds = xr.concat(model_var_avg_list, dim='member_id')#.sortby('plev').assign_coords({'plev':plevs})
        models_var_avg_list.append(model_var_avg_ds.assign_coords({'model':model}))

    models_var_avg_ds = xr.concat(models_var_avg_list, dim='model')
    #break
    models_var_avg_ds.to_netcdf(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{var}_WUSavg_19800101-20241231_allmodels.nc')

ts
GFDL-ESM4 3
IPSL-CM6A-LR 11
CNRM-CM6-1 6
MRI-ESM2-0 5
CNRM-ESM2-1 9
CanESM5 50
CanESM5-CanOE 3
UKESM1-0-LL 6
MIROC6 50
MPI-ESM1-2-LR 9
CESM2-WACCM 3
FGOALS-g3 4
MIROC-ES2L 16
NorESM2-LM 3
ACCESS-CM2 3
KACE-1-0-G 3
FIO-ESM-2-0 3
GISS-E2-1-G 14
EC-Earth3-Veg 7
EC-Earth3 19
CESM2 3
EC-Earth3-Veg-LR 3


In [66]:
models_var_avg_ds.to_netcdf(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{var}_WUSavg_19800101-20241231_allmodels.nc')

In [26]:
for var in ['rlds'][:]:    
    print(var)
    models_var_avg_list = []

    for model in list(modmembers_lwd.keys()):
        print(model, len(modmembers_lwd[model]))
        model_var_avg_list = []


        for member_id in modmembers_lwd[model][:]:
            ds_hist = xr.open_dataset(f'/d4/tessj/data/CMIP6/for_tess/{model}/rlds_Amon_{model}_historical_{member_id}_18500101-20141231_regrid.nc').sel(time=slice('1980-01-01', '2014-12-30'))
            ds_ss245 = xr.open_dataset(f'/d4/tessj/data/CMIP6/for_tess/{model}/rlds_Amon_{model}_ssp245_{member_id}_20150101-20241231_regrid.nc')
            ds_total = xr.concat([ds_hist, ds_ss245], dim='time')
            #break
            #if len(ds_total.time) != len(times): 
            #    print(model, member_id)

            landmask = landmask_era5.interp_like(ds_hist)
            landmask = landmask.where(landmask>0.4, np.nan)
            landmask = landmask/landmask
            weights = np.cos(np.deg2rad(landmask.lat))

            ds_wus = ds_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
            ds_wus_mean = ds_wus.weighted(weights).mean(['lon','lat'])
            model_var_avg_list.append(ds_wus_mean.assign_coords({'member_id':member_id, 'time':times}))

        model_var_avg_ds = xr.concat(model_var_avg_list, dim='member_id')#.sortby('plev').assign_coords({'plev':plevs})
        models_var_avg_list.append(model_var_avg_ds.assign_coords({'model':model}))

    models_var_avg_ds = xr.concat(models_var_avg_list, dim='model')
    #break
    models_var_avg_ds.to_netcdf(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{var}_WUSavg_19800101-20241231_allmodels.nc')

rlds
CNRM-CM6-1 6
IPSL-CM6A-LR 11
EC-Earth3-Veg 2
CanESM5-CanOE 3
MIROC6 46
MRI-ESM2-0 5
MPI-ESM1-2-LR 10
FIO-ESM-2-0 3
ACCESS-CM2 3
FGOALS-g3 4
EC-Earth3 6
CESM2 3
CESM2-WACCM 3
MIROC-ES2L 3
CNRM-ESM2-1 9
CanESM5 50
KACE-1-0-G 3
UKESM1-0-LL 6
GISS-E2-1-G 14


In [23]:
        sample_grid = xr.DataArray(coords = {'lat':(('lat'),np.arange(-89.5,89.6)),
                                     'lon':(('lon'),np.arange(-179.5,179.6))},
                           dims = ['lat','lon'])

In [27]:
for var in ['rldscs'][:]:    
    print(var)
    models_var_avg_list = []

    for model in list(modmembers_lwd.keys()):
        print(model, len(modmembers_rldscs[model]))
        model_var_avg_list = []


        for member_id in modmembers_rldscs[model][:]:
            ds_hist = xr.open_dataset(f'/d5/tessj/data/CMIP6/cmip6-esgf-globus/historical-processed/{model}/rldscs_Amon_{model}_historical_{member_id}_19500101-20141231.nc').sel(time=slice('1980-01-01', '2014-12-30'))
            ds_ss245 = xr.open_dataset(f'/d5/tessj/data/CMIP6/cmip6-esgf-globus/ssp245-processed/{model}/rldscs_Amon_{model}_ssp245_{member_id}_20150101-20241231.nc')
            ds_total = xr.concat([ds_hist, ds_ss245], dim='time')
            rgrd = xe.Regridder(ds_total,sample_grid,'bilinear',ignore_degenerate=True, periodic=True)
            ds_total = rgrd(ds_total)
            #break
            #if len(ds_total.time) != len(times): 
            #    print(model, member_id)

            landmask = landmask_era5.interp_like(ds_total)
            landmask = landmask.where(landmask>0.4, np.nan)
            landmask = landmask/landmask
            weights = np.cos(np.deg2rad(landmask.lat))

            ds_wus = ds_total.where(landmask==landmask).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2],wusbox[3]))
            ds_wus_mean = ds_wus.weighted(weights).mean(['lon','lat'])
            model_var_avg_list.append(ds_wus_mean.assign_coords({'member_id':member_id, 'time':times}))

        model_var_avg_ds = xr.concat(model_var_avg_list, dim='member_id')#.sortby('plev').assign_coords({'plev':plevs})
        models_var_avg_list.append(model_var_avg_ds.assign_coords({'model':model}))

    models_var_avg_ds = xr.concat(models_var_avg_list, dim='model')
    #break
    models_var_avg_ds.to_netcdf(f'/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/{var}_WUSavg_19800101-20241231_allmodels.nc')

rldscs
CNRM-CM6-1 6
IPSL-CM6A-LR 11
EC-Earth3-Veg 7
CanESM5-CanOE 3
MIROC6 50
MRI-ESM2-0 5
MPI-ESM1-2-LR 30
FIO-ESM-2-0 3
ACCESS-CM2 5
FGOALS-g3 4
EC-Earth3 19
CESM2 3
CESM2-WACCM 3
MIROC-ES2L 29
CNRM-ESM2-1 10
CanESM5 50
KACE-1-0-G 3
UKESM1-0-LL 6
GISS-E2-1-G 16
